In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1sDEyfEd07odh5rY25F5qSc4QyW1yJLhP?usp=drive_link

Retrieving folder contents
Retrieving folder 1FtvLBNczH4WR-WPiwGZs0-IihsnguZII Aya-23-8b
Retrieving folder 1Jv89hpEUIi5bX38KQ3btRWnd_UBgi81h 5k_sample
Processing file 1vgy_MURcAbr_xT1AuBsyaCpSY33miw01 aya-23-8B_clean_merged.csv
Retrieving folder 1-XHTgfSjL_ZrxDHHGR4E3m0e5rw7M7A0 100_sample
Processing file 1yV7dX7HaB6yaasOBbwo64R06m8eEu5Dxfl_DHib3UnI aya-23-8B_1
Processing file 1-LHMAobeP-hj-HyL9ov3Dq2V_HehdwVm aya-23-8B_1.csv
Processing file 1eMLha7jg6FrTsWTKyh6P_-bakYqD1a0c aya-23-8B_2.csv
Processing file 116m4PXAbry90d5rlQ1TtA0TtwYJEgdPS aya-23-8B_3.csv
Processing file 1WQlWlioSFh1rXJq-2haAjzrcb1mIZDE3 aya-23-8B_4.csv
Processing file 1zeFOCwshGKFnf0Ln6Mc-gQCopOrfsV07 aya-23-8B_5.csv
Retrieving folder 1YwAmTeRtvKpnv0b5e6OgnkuJ-23jS-dD DeepSeek-v3.1
Retrieving folder 1vqk0QP2ACdmS57ATnEUVzzPR4dUivgjD 5k_sample
Processing file 1eGpWlEgocXL84T4pGyronmtN6lu7Lur6 DeepSeek-V3.1_clean_merged.csv
Retrieving folder 1-Nk4UAJOFWexMci2EIABjMo8IvP8FDL2 100_sample
Processing file 1MTL6PrzAcxYvWXJ7l

In [ ]:
!pip install bert_score
!pip install underthesea
!pip install vncorenlp
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 24.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645933 sha256=827767beb6e8d49b74da642f911dab45758b6c5f091521313cef871e91e710dc
  Stored in directory: /root/.cache/pip/wheels/6f/19/20/ec7083125fd06db1a19d0d3ca18806ecf4e8ed1464713b4efa
Successfully built vncorenlp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


# evaluation

In [ ]:
# Clean Evaluation Pipeline for Vietnamese Text Similarity Metrics
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
# !pip install -q bert-score evaluate underthesea vncorenlp
# !pip install -q sentence-transformers editdistance nltk


# ============================================================
# IMPORTS
# ============================================================
import os
from collections import Counter

import editdistance
import numpy as np
import pandas as pd
from evaluate import load
from nltk.translate.meteor_score import meteor_score
from sentence_transformers import CrossEncoder
from tqdm import tqdm
from underthesea import word_tokenize


# ============================================================
# TOKENIZATION
# ============================================================
def tokenize_vietnamese(text: str):
    """
    Vietnamese word segmentation.
    """

    if pd.isna(text):
        return []

    text = str(text).strip().lower()

    if not text:
        return []

    return word_tokenize(text, format="text").split()


# ============================================================
# NGRAM F1
# ============================================================
def build_ngrams(tokens, n):
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]



def ngram_f1(reference_tokens, candidate_tokens, n=3):
    """
    Compute n-gram F1 score.
    """

    if len(reference_tokens) < n or len(candidate_tokens) < n:
        return 0.0

    reference_ngrams = Counter(build_ngrams(reference_tokens, n))
    candidate_ngrams = Counter(build_ngrams(candidate_tokens, n))

    overlap = sum((reference_ngrams & candidate_ngrams).values())

    if overlap == 0:
        return 0.0

    precision = overlap / max(sum(candidate_ngrams.values()), 1)
    recall = overlap / max(sum(reference_ngrams.values()), 1)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


# ============================================================
# EDIT DISTANCE
# ============================================================
def compute_edit_distance(reference_text, candidate_text):
    return editdistance.eval(reference_text, candidate_text)



def compute_edit_similarity(reference_text, candidate_text):
    """
    Normalized edit similarity.
    Value range: [0, 1]
    """

    max_length = max(len(reference_text), len(candidate_text))

    if max_length == 0:
        return 1.0

    distance = compute_edit_distance(reference_text, candidate_text)

    return 1 - (distance / max_length)


# ============================================================
# METEOR
# ============================================================
def compute_meteor(reference_tokens, candidate_tokens):
    try:
        return meteor_score([reference_tokens], candidate_tokens)
    except:
        return np.nan


# ============================================================
# CROSS ENCODER
# ============================================================
def load_cross_encoder(model_name="BAAI/bge-reranker-v2-m3"):
    return CrossEncoder(model_name, max_length=512)


# ============================================================
# MAIN EVALUATION FUNCTION
# ============================================================
def evaluate_dataframe(
    dataframe,
    text_column="text",
    prediction_column="paraphrase_r3",
    bert_model="bert-base-multilingual-cased",
    ngram_size=3,
    cross_encoder_model="BAAI/bge-reranker-v2-m3",
):
    """
    Compute:
        - METEOR
        - BERTScore
        - N-gram F1
        - Edit Distance
        - Edit Similarity
        - Cross Encoder Score
    """

    df = dataframe.copy()

    # ========================================================
    # CLEAN TEXT
    # ========================================================
    df[text_column] = (
        df[text_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df[prediction_column] = (
        df[prediction_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    references = df[text_column].tolist()
    candidates = df[prediction_column].tolist()


    # ========================================================
    # TOKENIZATION
    # ========================================================
    reference_tokens = [tokenize_vietnamese(text) for text in references]
    candidate_tokens = [tokenize_vietnamese(text) for text in candidates]


    # ========================================================
    # METEOR
    # ========================================================
    print("Computing METEOR...")

    meteor_scores = [
        compute_meteor(ref_tok, cand_tok)
        for ref_tok, cand_tok in tqdm(
            zip(reference_tokens, candidate_tokens),
            total=len(df),
        )
    ]


    # ========================================================
    # BERTSCORE
    # ========================================================
    print("Computing BERTScore...")

    bertscore_metric = load("bertscore")

    bertscore_result = bertscore_metric.compute(
        predictions=candidates,
        references=references,
        lang="vi",
        model_type=bert_model,
    )

    bert_scores = bertscore_result["f1"]


    # ========================================================
    # NGRAM F1
    # ========================================================
    print("Computing N-gram F1...")

    ngram_scores = [
        ngram_f1(ref_tok, cand_tok, n=ngram_size)
        for ref_tok, cand_tok in tqdm(
            zip(reference_tokens, candidate_tokens),
            total=len(df),
        )
    ]


    # ========================================================
    # EDIT DISTANCE
    # ========================================================
    print("Computing Edit Distance...")

    edit_distances = [
        compute_edit_distance(ref, cand)
        for ref, cand in tqdm(
            zip(references, candidates),
            total=len(df),
        )
    ]


    # ========================================================
    # EDIT SIMILARITY
    # ========================================================
    print("Computing Edit Similarity...")

    edit_similarities = [
        compute_edit_similarity(ref, cand)
        for ref, cand in tqdm(
            zip(references, candidates),
            total=len(df),
        )
    ]


    # ========================================================
    # CROSS ENCODER SCORE
    # ========================================================
    print("Loading Cross Encoder...")

    cross_encoder = load_cross_encoder(cross_encoder_model)

    print("Computing Cross Encoder Score...")

    sentence_pairs = list(zip(references, candidates))

    cross_encoder_scores = cross_encoder.predict(
        sentence_pairs,
        show_progress_bar=True,
        batch_size=32,
    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================
    df["meteor"] = meteor_scores
    df["bertscore"] = bert_scores
    df["ngram"] = ngram_scores
    df["editdistance"] = edit_distances
    df["edit_similarity"] = edit_similarities
    df["crossencoder_score"] = cross_encoder_scores

    return df


# ============================================================
# SUMMARY FUNCTION
# ============================================================
def summarize_metrics(dataframe):
    metric_columns = [
        "meteor",
        "bertscore",
        "ngram",
        "editdistance",
        "edit_similarity",
        "crossencoder_score",
    ]

    summary = dataframe[metric_columns].mean().round(4)

    return summary.to_frame(name="mean_score")


# ============================================================
# EXAMPLE USAGE
# ============================================================
if __name__ == "__main__":

        # ============================================================
    # DATASET PATHS
    # ============================================================

    DATASETS = {
        "aya": "./llm_generated_data/Aya-23-8b/5k_sample/aya-23-8B_clean_merged.csv",

        "deepseek": "./llm_generated_data/DeepSeek-v3.1/5k_sample/DeepSeek-V3.1_clean_merged.csv",

        "gemma": "./llm_generated_data/Gemma2-9B-it/5k sample/output_gemma_clean_5k.csv",

        "gpt54mini": "./llm_generated_data/Gpt-5.4-mini/5k/gpt-5.4-mini_clean_merged.csv",

        "sailor": "./llm_generated_data/Sailor2-8B/5k_sample/output_sailor_clean_5k.csv",
    }


    # ============================================================
    # EVALUATION LOOP
    # ============================================================

    all_summaries = []

    for model_name, path in DATASETS.items():

        print(f"\n{'=' * 60}")
        print(f"Evaluating: {model_name}")
        print(f"{'=' * 60}")

        # --------------------------------------------------------
        # LOAD DATA
        # --------------------------------------------------------
        df = pd.read_csv(path)

        print(f"Loaded {len(df)} rows")


        # --------------------------------------------------------
        # CHECK REQUIRED COLUMNS
        # --------------------------------------------------------
        required_columns = ["text", "paraphrase_r3"]

        missing_columns = [
            col for col in required_columns
            if col not in df.columns
        ]

        if len(missing_columns) > 0:

            print(f"Missing columns: {missing_columns}")
            print(f"Available columns: {df.columns.tolist()}")

            continue


        # --------------------------------------------------------
        # EVALUATE
        # --------------------------------------------------------
        result_df = evaluate_dataframe(
            dataframe=df,
            text_column="text",
            prediction_column="paraphrase_r3",
        )


        # --------------------------------------------------------
        # SAVE RESULT
        # --------------------------------------------------------
        output_path = f"{model_name}_metrics.csv"

        result_df.to_csv(
            output_path,
            index=False
        )

        print(f"Saved: {output_path}")


        # --------------------------------------------------------
        # SUMMARY
        # --------------------------------------------------------
        summary_df = summarize_metrics(result_df)

        summary_df.columns = [model_name]

        all_summaries.append(summary_df)


    # ============================================================
    # FINAL SUMMARY TABLE
    # ============================================================

    final_summary = pd.concat(
        all_summaries,
        axis=1
    )

    print("\n")
    print("=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)

    print(final_summary)


Evaluating: aya
Loaded 4724 rows
Computing METEOR...


100%|██████████| 4724/4724 [00:05<00:00, 883.45it/s] 


Computing BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

Loading Cross Encoder...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Computing Cross Encoder Score...


Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Saved: sailor_metrics.csv


FINAL SUMMARY
                        aya  deepseek    gemma  gpt54mini    sailor
meteor               0.2643    0.2299   0.1640     0.4846    0.1377
bertscore            0.7356    0.7308   0.7071     0.7951    0.6883
ngram                0.0455    0.0239   0.0147     0.1059    0.0069
editdistance        78.1933   82.9667  82.8579    62.3808  101.7088
edit_similarity      0.3511    0.3255   0.2887     0.4919    0.2540
crossencoder_score   0.6613    0.6943   0.5365     0.9303    0.5644
